In [1]:
import torch
import stable_worldmodel as swm
from model import WorldModel

In [2]:
device = 'mps'

In [3]:
dataset = swm.data.load_dataset(
    'tutorial_pusht.lance',
    num_steps=8,
    frameskip=1,
    keys_to_load=['pixels', 'action', 'state'],
)

In [4]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    drop_last=True,
)

batch = next(iter(loader))
print(batch['pixels'].shape)  # (B, T, C, H, W)
print(batch['action'].shape)  # (B, T, action_dim)


torch.Size([64, 8, 3, 96, 96])
torch.Size([64, 8, 2])


In [5]:
batch['action'] = torch.nan_to_num(batch['action'], 0.0)

In [6]:
model = WorldModel(
    hidden_dim=192, 
    patch_size=12,
    channels=3, 
    enc_nb_heads=3, 
    enc_nb_layers=12, 
    height=96, 
    width=96, 
    pred_nb_layers=6, 
    pred_p_dropout=0.1, 
    pred_nb_heads=16, 
    action_dim=2
).to(device)

In [7]:
for batch in loader:
    frames = batch['pixels'].to(device)
    actions = batch['action'].to(device)
    pred = model(frames, actions)
    print(pred.shape)

RuntimeError: MPS device does not support linear for non-float inputs